# Caption keyframe bằng Qwen2.5-VL trên T4 x2

Sinh **mô tả tiếng Việt + nhãn vật thể** cho ảnh keyframe đã tách sẵn của một
batch (`L21_a`, `L22_a`, …), ghi ra stage pack đúng `contracts/stage_pack.schema.json`.

**Input:** `Keyframes_<BATCH>/keyframes/<video_id>/*.jpg` trong Kaggle Dataset.
**Output:** `manifests/caption_manifest.jsonl`, `manifests/caption_frame_status.jsonl`,
`videos/<video_id>/caption_manifest.jsonl`, `model_info.json`, `_SUCCESS.json`, zip.

## Hai quyết định thiết kế cần biết trước khi sửa notebook

**1. Trả lời hai dòng `CAPTION:` / `OBJECTS:`, không phải JSON.** Bản 3B thỉnh
thoảng trả JSON cụt giữa mảng — `scripts/enrich_keyframes_fpt.py` gặp đúng lỗi
này ở 3/307 frame lần chạy full đầu tiên, luôn do mảng dài. Hai dòng có tiền tố
thì cụt dòng sau vẫn giữ được dòng trước, và tiền tố ASCII không phụ thuộc việc
model gõ đúng dấu tiếng Việt.

**2. Vật thể ghi thành nhãn, không kèm bbox.** `objectRow` của contract bắt buộc
có `bbox`, mà bắt model đoán toạ độ thì vừa sai vừa kéo chất lượng phần chữ
xuống (đo được ở lượt OCR). Nên nhãn vật thể đi vào `caption_manifest.jsonl`
dưới dạng `caption_type: "tags"` — hợp lệ với schema, hữu ích cho BM25, và
không bịa ra một toạ độ nào. Cần bbox thật thì dùng detector chuyên dụng
(OWLv2) ở một stage riêng.

## Thứ tự tác vụ

1. `discover_keyframes` → 2. `download_weights` → 3. `smoke_test` →
4. `caption_all` → 5. `merge_shards` → 6. `package_zip` / `validate_output`

In [ ]:
# Kaggle đã có sẵn torch bản CUDA — cài lại torch từ PyPI có thể kéo về bản
# CPU-only và làm hỏng GPU cho cả phiên. Chỉ nâng transformers + accelerate.
%pip install -q --no-cache-dir "transformers>=4.51,<5" "accelerate>=0.34"

## 1. Cấu hình

Chạy batch khác thì **chỉ đổi `BATCH`**.

So với notebook OCR: `MAX_VISUAL_TOKENS` thấp hơn (512 thay vì 1024) và
`BATCH_SIZE` cao hơn (4 thay vì 2). Mô tả cảnh không cần đọc chữ nhỏ, nên hạ độ
phân giải đổi lấy thông lượng là đổi có lời.

In [ ]:
from pathlib import Path
import os

# ===================== ĐỔI DUY NHẤT DÒNG NÀY =====================
BATCH = "L21_a"          # L21_a, L21_b, L22_a, ...
# =================================================================

# Để trống = tự dò `Keyframes_<BATCH>` trong /kaggle/input.
KEYFRAME_ROOT_OVERRIDE = os.environ.get("AIC_KEYFRAME_ROOT", "")

STAGE_NAME = f"caption_{BATCH}"
OUTPUT_ROOT = Path(f"/kaggle/working/aic_stage_caption_{BATCH}")
ZIP_PATH = Path(f"/kaggle/working/caption_{BATCH}_output.zip")

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DTYPE = "float16"          # T4 là Turing: không có tensor core bf16, fp16 nhanh và ổn định hơn
BATCH_SIZE = 4             # ảnh mỗi lần generate, TRÊN MỖI GPU. OOM thì worker tự chia đôi.
MAX_VISUAL_TOKENS = 512    # 1 visual token phủ 28x28 px
MIN_VISUAL_TOKENS = 128
MAX_NEW_TOKENS = 200

MAX_FRAMES = 0             # 0 = tất cả; đặt số nhỏ để thử nhanh
SMOKE_FRAMES = 4           # số frame MỖI GPU cho lần chạy thử ở mục 6
PACK_VERSION = "1.0.0"

print("BATCH       =", BATCH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)

## 2. Theo dõi tiến độ & đo thời gian

- `TaskTracker.stage(name)` — log bắt đầu/kết thúc/lỗi, đo thời gian, ghi `stage_timings.jsonl`.
- `progress.json` — trạng thái hiện tại (tác vụ, số frame xong, ETA), ghi atomic nên đọc từ tab khác luôn thấy file hợp lệ.
- `progress.log` — log dạng text, có nhịp heartbeat để biết notebook chưa treo.

In [ ]:
SHOW_TQDM = True
POLL_INTERVAL_SEC = 3          # nhịp đọc file tiến độ của worker
HEARTBEAT_INTERVAL_SEC = 30    # nhịp ghi một dòng log tổng hợp
WRITE_PROGRESS_JSON = True

PROGRESS_JSON_PATH = OUTPUT_ROOT / "progress.json"
PROGRESS_LOG_PATH = OUTPUT_ROOT / "progress.log"
STAGE_TIMINGS_PATH = OUTPUT_ROOT / "stage_timings.jsonl"
SHARD_DIR = OUTPUT_ROOT / "shards"
TASKS_PATH = OUTPUT_ROOT / "tasks.jsonl"

print("OUTPUT_ROOT        =", OUTPUT_ROOT)
print("PROGRESS_JSON_PATH =", PROGRESS_JSON_PATH)

In [ ]:
import json
import os
import subprocess
import sys
import time
import zipfile
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Optional

from tqdm.auto import tqdm


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_progress_log(message: str) -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    line = f"[{utc_now()}] {message}"
    print(line, flush=True)
    with PROGRESS_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temp, path)


def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_bytes(path, (json.dumps(payload, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))


def atomic_write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    content = "".join(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n" for row in rows)
    atomic_write_bytes(path, content.encode("utf-8"))


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def read_json_safe(path: Path) -> dict[str, Any]:
    """Đọc file tiến độ của worker; file chưa kịp tạo thì trả dict rỗng."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}


def write_progress_state(payload: dict[str, Any]) -> None:
    if not WRITE_PROGRESS_JSON:
        return
    atomic_write_json(PROGRESS_JSON_PATH, {"stage": STAGE_NAME, "updated_at_utc": utc_now(), **payload})


# ---------------------------------------------------------------------------
# TaskTracker: mỗi tác vụ là một "stage" có tên, có start/end/duration
# ---------------------------------------------------------------------------
@dataclass
class StageRecord:
    name: str
    start_ts: float
    end_ts: Optional[float] = None
    status: str = "running"
    detail: str = ""

    @property
    def duration_sec(self) -> float:
        end = self.end_ts if self.end_ts is not None else time.perf_counter()
        return end - self.start_ts


class TaskTracker:
    """Theo dõi tiến độ & thời gian chạy của từng tác vụ trong pipeline."""

    def __init__(self) -> None:
        self.records: list[StageRecord] = []
        self._pipeline_started = time.perf_counter()

    @contextmanager
    def stage(self, name: str, detail: str = ""):
        record = StageRecord(name=name, start_ts=time.perf_counter(), detail=detail)
        self.records.append(record)
        append_progress_log(f"▶ BẮT ĐẦU  [{name}]" + (f" — {detail}" if detail else ""))
        write_progress_state({"status": "running", "current_task": name,
                              "elapsed_sec": round(time.perf_counter() - self._pipeline_started, 3)})
        try:
            yield record
            record.status = "success"
        except Exception as exc:
            record.status = "failed"
            record.end_ts = time.perf_counter()
            append_progress_log(f"✖ LỖI      [{name}] sau {record.duration_sec:.1f}s | {exc!r}")
            append_jsonl(STAGE_TIMINGS_PATH, self._record_to_row(record))
            write_progress_state({"status": "failed", "current_task": name, "error": repr(exc)})
            raise
        else:
            record.end_ts = time.perf_counter()
            suffix = f" — {record.detail}" if record.detail else ""
            append_progress_log(f"✔ HOÀN TẤT [{name}] sau {record.duration_sec:.1f}s{suffix}")
            append_jsonl(STAGE_TIMINGS_PATH, self._record_to_row(record))

    @staticmethod
    def _record_to_row(record: StageRecord) -> dict[str, Any]:
        return {
            "task": record.name,
            "status": record.status,
            "duration_sec": round(record.duration_sec, 3),
            "detail": record.detail,
            "finished_at_utc": utc_now(),
        }

    def elapsed_total_sec(self) -> float:
        return time.perf_counter() - self._pipeline_started


def create_zip(folder: Path, zip_path: Path) -> Path:
    temp = zip_path.with_name(zip_path.name + ".tmp")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(folder))
    os.replace(temp, zip_path)
    return zip_path


def detect_gpus() -> int:
    """Đếm GPU qua nvidia-smi, KHÔNG qua torch.

    Quan trọng: notebook cha không được khởi tạo CUDA context. Gọi
    `torch.cuda.*` ở đây sẽ chiếm sẵn vài trăm MB trên GPU 0 — phần bộ nhớ đó
    lẽ ra thuộc về worker.
    """
    try:
        result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.SubprocessError):
        return 0
    return sum(1 for line in result.stdout.splitlines() if line.startswith("GPU "))

## 3. Tìm thư mục keyframe & kiểm tra trước khi tốn GPU

Cell dưới in bảng từng video kèm cảnh báo. Đọc bảng này **trước khi** chạy tiếp:

- `video_id` không khớp `^L\d{2}_V\d{3}$` → `assemble.py` sẽ từ chối pack.
- `frame_idx` chạy liên tiếp `0..n-1` → tên file nhiều khả năng là *số thứ tự ảnh*
  chứ không phải *chỉ số frame thật*. Khoá join của stage pack là
  `(video_id, frame_idx)`, nên trường hợp này phải ánh xạ lại theo `image_relpath` lúc merge.
- Trùng `(video_id, frame_idx)` → dừng hẳn, vì manifest sẽ mơ hồ.

In [ ]:
import re

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
VIDEO_ID_RE = re.compile(r"^L\d{2}_V\d{3}$")
DIGITS_RE = re.compile(r"\d+")


def resolve_keyframe_root(batch: str) -> Path:
    """Tìm thư mục keyframe của batch trong /kaggle/input.

    Kaggle mount dataset theo slug do nó tự đặt, không theo tên bạn gõ lúc
    upload — nên notebook DÒ thay vì bắt bạn đoán đúng đường dẫn.
    """
    if KEYFRAME_ROOT_OVERRIDE:
        root = Path(KEYFRAME_ROOT_OVERRIDE)
        if not root.is_dir():
            raise FileNotFoundError(f"KEYFRAME_ROOT_OVERRIDE không tồn tại: {root}")
        return root

    input_root = Path("/kaggle/input")
    if not input_root.is_dir():
        raise FileNotFoundError("Không thấy /kaggle/input — notebook này chạy trên Kaggle.")

    target = f"Keyframes_{batch}"
    matches: list[Path] = []
    for pattern in (target, f"*/{target}", f"*/*/{target}", f"*/*/*/{target}", f"*/*/*/*/{target}"):
        matches.extend(path for path in sorted(input_root.glob(pattern)) if path.is_dir())
        if matches:
            break

    if not matches:
        available = sorted(path.name for path in input_root.glob("*/*") if path.is_dir())[:40]
        raise FileNotFoundError(
            f"Không tìm thấy thư mục '{target}' dưới /kaggle/input.\n"
            f"Kiểm tra lại BATCH, hoặc đặt KEYFRAME_ROOT_OVERRIDE.\n"
            f"Một số thư mục đang có: {available}"
        )
    if len(matches) > 1:
        print(f"[CẢNH BÁO] có {len(matches)} thư mục tên '{target}', dùng cái đầu tiên:")
        for path in matches:
            print("   ", path)

    root = matches[0]
    nested = root / "keyframes"
    return nested if nested.is_dir() else root


def parse_video_id(image_path: Path, root: Path) -> str:
    parts = image_path.relative_to(root).parts[:-1]
    for part in reversed(parts):
        if VIDEO_ID_RE.match(part):
            return part
    return parts[-1] if parts else ""


def parse_frame_idx(image_path: Path) -> int:
    """Cụm số CUỐI trong tên file. `frame_000090` -> 90, `L21_V001_000090` -> 90."""
    digits = DIGITS_RE.findall(image_path.stem)
    return int(digits[-1]) if digits else -1


def scan_keyframes(root: Path) -> list[dict[str, Any]]:
    tasks = []
    for image_path in root.rglob("*"):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        relpath = image_path.relative_to(root).as_posix()
        tasks.append({
            "key": relpath,  # khoá resume: đường dẫn tương đối là duy nhất tuyệt đối
            "video_id": parse_video_id(image_path, root),
            "frame_idx": parse_frame_idx(image_path),
            "image_relpath": relpath,
            "image_path": str(image_path),
        })
    # Sắp theo (video, frame_idx) chứ không theo tên file: sắp theo tên thì
    # frame_9 đứng sau frame_10 và thứ tự trong manifest sẽ vô nghĩa.
    tasks.sort(key=lambda task: (task["video_id"], task["frame_idx"], task["image_relpath"]))
    return tasks

In [ ]:
from collections import Counter, defaultdict

tracker = TaskTracker()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
if STAGE_TIMINGS_PATH.exists():
    STAGE_TIMINGS_PATH.unlink()

with tracker.stage("discover_keyframes") as rec:
    KEYFRAME_ROOT = resolve_keyframe_root(BATCH)
    print("KEYFRAME_ROOT =", KEYFRAME_ROOT)
    tasks = scan_keyframes(KEYFRAME_ROOT)
    if not tasks:
        raise FileNotFoundError(f"Không có ảnh nào dưới {KEYFRAME_ROOT}")
    if MAX_FRAMES > 0:
        tasks = tasks[:MAX_FRAMES]
    atomic_write_jsonl(TASKS_PATH, tasks)
    rec.detail = f"{len(tasks)} keyframe"

by_video: dict[str, list[dict[str, Any]]] = defaultdict(list)
for task in tasks:
    by_video[task["video_id"]].append(task)

print(f"\n{len(tasks)} keyframe / {len(by_video)} video\n")
print(f"{'video_id':16s} {'ảnh':>6s} {'frame_idx nhỏ nhất':>19s} {'lớn nhất':>10s}  ghi chú")

bad_video_id, no_digit, ordinal_like = [], [], []
for video_id in sorted(by_video):
    rows = by_video[video_id]
    indices = [row["frame_idx"] for row in rows]
    notes = []
    if not VIDEO_ID_RE.match(video_id):
        notes.append("video_id KHÔNG khớp ^L##_V###$")
        bad_video_id.append(video_id)
    if any(index < 0 for index in indices):
        notes.append("có file không chứa số")
        no_digit.append(video_id)
    # Nếu chỉ số chạy liền 0..n-1 hoặc 1..n thì nhiều khả năng đó là SỐ THỨ TỰ
    # ảnh, không phải chỉ số frame thật. Join (video_id, frame_idx) lúc assemble
    # sẽ trỏ sai frame — phải biết trước khi đốt hàng giờ GPU.
    unique = set(indices)
    if len(unique) == len(rows) and unique in (set(range(len(rows))), set(range(1, len(rows) + 1))):
        notes.append("chỉ số liên tiếp -> có thể là SỐ THỨ TỰ, không phải frame thật")
        ordinal_like.append(video_id)
    print(f"{video_id:16s} {len(rows):6d} {min(indices):19d} {max(indices):10d}  {'; '.join(notes)}")

duplicates = [key for key, count in Counter((t["video_id"], t["frame_idx"]) for t in tasks).items() if count > 1]

print()
if bad_video_id:
    print(f"[CẢNH BÁO] {len(bad_video_id)} video_id không đúng dạng L##_V### -> assemble sẽ từ chối: {bad_video_id[:10]}")
if no_digit:
    print(f"[CẢNH BÁO] {len(no_digit)} video có file không suy ra được frame_idx: {no_digit[:10]}")
if ordinal_like:
    print(f"[CẢNH BÁO] {len(ordinal_like)} video có frame_idx trông như số thứ tự: {ordinal_like[:10]}")
    print("           Vẫn chạy được, nhưng khi merge về dataset canonical phải ánh xạ lại theo image_relpath.")
if duplicates:
    raise ValueError(
        f"{len(duplicates)} cặp (video_id, frame_idx) bị TRÙNG, ví dụ {duplicates[:5]}. "
        "Khoá join của stage pack là (video_id, frame_idx) nên trùng là hỏng manifest. "
        "Sửa tên file hoặc đặt KEYFRAME_ROOT_OVERRIDE trỏ đúng thư mục trước khi chạy tiếp."
    )
if not (bad_video_id or no_digit or ordinal_like):
    print("Preflight sạch: video_id đúng dạng, frame_idx suy ra được, không trùng khoá.")

## 4. Worker caption

Ghi ra `/kaggle/working/aic_caption_worker.py`. Mỗi worker là một process độc
lập giữ một model trên một GPU, nhận `--shard i --num-shards N` và chỉ xử lý
`tasks[i::N]`.

Hàm `parse_answer` còn cắt bỏ mấy cụm mở đầu vô nghĩa mà model hay thêm ("bức
ảnh này cho thấy…") vì chúng làm loãng BM25. Nó **không** cắt từ "Anh" đứng một
mình — "Anh Nam đang phỏng vấn" là caption hợp lệ.

In [ ]:
%%writefile /kaggle/working/aic_caption_worker.py
"""Worker caption — mỗi process giữ một model trên MỘT GPU.

Cùng kiến trúc với worker OCR (shard theo process, append từng dòng, resume
được), khác ở prompt và ở cách phân tích output.

Định dạng trả lời là hai dòng có tiền tố ASCII `CAPTION:` / `OBJECTS:` chứ
không phải JSON. Lý do đo được: bản 3B thỉnh thoảng trả JSON cụt giữa mảng
(scripts/enrich_keyframes_fpt.py gặp ở 3/307 frame khi chạy full lần đầu, luôn
là do mảng dài). Hai dòng có tiền tố thì cụt dòng sau vẫn giữ được dòng trước,
và tiền tố ASCII thì không phụ thuộc model gõ đúng dấu tiếng Việt.
"""

from __future__ import annotations

import argparse
import json
import os
import re
import time
import traceback
from pathlib import Path

PROMPT = """Mô tả khung hình video này bằng tiếng Việt.

Trả lời ĐÚNG hai dòng theo mẫu sau, không thêm gì khác:
CAPTION: <1-2 câu mô tả khách quan những gì NHÌN THẤY: người, hành động, bối cảnh, vật thể nổi bật>
OBJECTS: <danh sách nhãn vật thể/người nhìn thấy rõ, cách nhau bằng dấu phẩy>

Quy tắc:
- Chỉ mô tả thứ THỰC SỰ có trong ảnh. Không suy đoán, không kể chuyện, không bình luận.
- Không nhắc đến "bức ảnh", "khung hình", "hình này" trong câu mô tả.
- OBJECTS chỉ ghi nhãn ngắn (ví dụ: người dẫn chương trình, micro, bàn làm việc). Không mô tả dài.
- Nếu không nhìn rõ vật thể nào, ghi: OBJECTS: khong-ro"""

NO_OBJECT = "khong-ro"
MAX_TAGS = 15
MAX_TAG_WORDS = 6

# Model hay mở đầu bằng đúng những cụm prompt đã cấm; cắt ở khâu phân tích cho
# chắc. Chỉ khớp cụm NHIỀU TỪ — không bao giờ khớp mỗi "ảnh"/"anh" đứng một
# mình, vì "Anh Nam đang..." là caption hợp lệ và cắt "Anh" là làm hỏng nghĩa.
CAPTION_NOISE = re.compile(
    r"^\s*(bức ảnh|buc anh|hình ảnh|hinh anh|khung hình|khung hinh|trong ảnh|trong anh)"
    r"(\s+(này|nay|đó|do|trên|tren))?"
    r"(\s+(cho thấy|cho thay|mô tả|mo ta|miêu tả|mieu ta|thể hiện|the hien|ghi lại|ghi lai|là|la))?"
    r"\s*[:,\-]?\s*",
    re.IGNORECASE,
)


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    os.replace(temp, path)


def parse_answer(text: str) -> tuple[str, list[str]]:
    """Trả `(caption, tags)`. Thiếu tiền tố thì lấy đoạn văn đầu làm caption."""

    caption = ""
    tag_line = ""
    leftover: list[str] = []
    for raw in text.splitlines():
        line = raw.strip().lstrip("-*•").strip()
        if not line:
            continue
        upper = line.upper()
        if upper.startswith("CAPTION:"):
            caption = caption or line.split(":", 1)[1].strip()
        elif upper.startswith("OBJECTS:"):
            tag_line = tag_line or line.split(":", 1)[1].strip()
        else:
            leftover.append(line)

    if not caption and leftover:
        caption = leftover[0]
    caption = CAPTION_NOISE.sub("", caption).strip().strip('"').strip()
    if caption:
        caption = caption[0].upper() + caption[1:]

    tags: list[str] = []
    for chunk in re.split(r"[,;/]| - ", tag_line):
        tag = chunk.strip().strip('"').strip(".").strip()
        if len(tag) < 2 or tag.lower() == NO_OBJECT:
            continue
        if len(tag.split()) > MAX_TAG_WORDS:
            continue
        if tag.lower() not in {existing.lower() for existing in tags}:
            tags.append(tag)
    return caption, tags[:MAX_TAGS]


def main() -> None:
    parser = argparse.ArgumentParser(description="Caption một shard keyframe trên một GPU")
    parser.add_argument("--tasks", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--progress", type=Path, required=True)
    parser.add_argument("--shard", type=int, default=0)
    parser.add_argument("--num-shards", type=int, default=1)
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--dtype", default="float16", choices=["float16", "bfloat16", "float32"])
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--max-visual-tokens", type=int, default=512)
    parser.add_argument("--min-visual-tokens", type=int, default=128)
    parser.add_argument("--max-new-tokens", type=int, default=200)
    parser.add_argument("--limit", type=int, default=0)
    args = parser.parse_args()

    tasks = read_jsonl(args.tasks)[args.shard :: args.num_shards]
    if args.limit > 0:
        tasks = tasks[: args.limit]

    args.out.parent.mkdir(parents=True, exist_ok=True)
    done_keys = set()
    if args.out.exists():
        for row in read_jsonl(args.out):
            done_keys.add(row["key"])
    todo = [task for task in tasks if task["key"] not in done_keys]

    state = {
        "shard": args.shard,
        "total": len(tasks),
        "done": len(tasks) - len(todo),
        "found": 0,
        "failed": 0,
        "status": "loading_model",
        "updated_at": time.time(),
    }
    write_json_atomic(args.progress, state)
    log(f"shard {args.shard}: {len(tasks)} frame, đã xong {state['done']}, còn {len(todo)}")
    if not todo:
        state["status"] = "success"
        write_json_atomic(args.progress, state)
        return

    import torch
    from PIL import Image, ImageFile
    from transformers import AutoProcessor

    ImageFile.LOAD_TRUNCATED_IMAGES = True

    try:
        from transformers import Qwen2_5_VLForConditionalGeneration as VLModel
    except ImportError:
        from transformers import AutoModelForImageTextToText as VLModel

    torch_dtype = getattr(torch, args.dtype)
    patch_area = 28 * 28

    processor = AutoProcessor.from_pretrained(
        args.model,
        min_pixels=args.min_visual_tokens * patch_area,
        max_pixels=args.max_visual_tokens * patch_area,
    )
    processor.tokenizer.padding_side = "left"

    model = VLModel.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model.to("cuda:0")
    model.eval()
    actual_dtype = next(model.parameters()).dtype
    log(f"shard {args.shard}: model sẵn sàng, dtype thực tế = {actual_dtype}")
    if actual_dtype != torch_dtype:
        log(f"shard {args.shard}: CẢNH BÁO dtype không như yêu cầu ({args.dtype})")

    chat_text = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}],
        tokenize=False,
        add_generation_prompt=True,
    )

    def load_image(path: Path):
        with Image.open(path) as handle:
            return handle.convert("RGB")

    def generate(images: list) -> list[str]:
        try:
            inputs = processor(
                text=[chat_text] * len(images), images=images, return_tensors="pt", padding=True
            ).to(model.device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=args.max_new_tokens,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.pad_token_id,
                )
            trimmed = generated[:, inputs["input_ids"].shape[1] :]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(images) == 1:
                raise
            middle = len(images) // 2
            log(f"shard {args.shard}: OOM ở batch {len(images)}, chia đôi và thử lại")
            return generate(images[:middle]) + generate(images[middle:])

    handle = args.out.open("a", encoding="utf-8")
    started = time.perf_counter()

    def emit(task: dict, caption: str, tags: list[str], status: str, error: str = "") -> None:
        record = {
            "key": task["key"],
            "video_id": task["video_id"],
            "frame_idx": task["frame_idx"],
            "image_relpath": task["image_relpath"],
            "caption": caption,
            "tags": tags,
            "status": status,
            "error": error,
        }
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()

    try:
        for start in range(0, len(todo), args.batch_size):
            batch = todo[start : start + args.batch_size]
            images, kept = [], []
            for task in batch:
                try:
                    images.append(load_image(Path(task["image_path"])))
                    kept.append(task)
                except Exception as exc:
                    state["failed"] += 1
                    state["done"] += 1
                    emit(task, "", [], "error", f"{type(exc).__name__}: {exc}")

            if kept:
                try:
                    outputs = generate(images)
                except Exception as exc:
                    outputs = None
                    log(f"shard {args.shard}: batch lỗi — {type(exc).__name__}: {exc}")
                    for task in kept:
                        state["failed"] += 1
                        state["done"] += 1
                        emit(task, "", [], "error", f"{type(exc).__name__}: {exc}")
                if outputs is not None:
                    for task, text in zip(kept, outputs):
                        caption, tags = parse_answer(text)
                        state["done"] += 1
                        if caption:
                            state["found"] += 1
                        # Caption rỗng là kết quả BẤT THƯỜNG (khác OCR, ảnh nào
                        # cũng phải mô tả được), nên đánh dấu empty để cell kiểm
                        # tra cuối notebook đếm ra và bạn biết mà chạy lại.
                        emit(task, caption, tags, "ok" if caption else "empty")

            elapsed = time.perf_counter() - started
            processed = max(1, state["done"] - (len(tasks) - len(todo)))
            state["status"] = "running"
            state["rate_sec_per_frame"] = round(elapsed / processed, 3)
            state["updated_at"] = time.time()
            write_json_atomic(args.progress, state)
    finally:
        handle.close()

    state["status"] = "success"
    state["updated_at"] = time.time()
    write_json_atomic(args.progress, state)
    log(f"shard {args.shard}: xong {state['done']}/{state['total']}, "
        f"có caption {state['found']}, lỗi {state['failed']}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        traceback.print_exc()
        raise SystemExit(1)

## 5. Đếm GPU & tải trọng số một lần

In [ ]:
from huggingface_hub import snapshot_download

NUM_GPUS = detect_gpus()
print("GPU phát hiện được:", NUM_GPUS)
if NUM_GPUS == 0:
    raise SystemExit(
        "Không thấy GPU. Vào Notebook Settings -> Accelerator -> 'GPU T4 x2' rồi chạy lại."
    )
NUM_SHARDS = NUM_GPUS

# Tải TRƯỚC ở tiến trình cha. Để hai worker cùng tải một repo là để chúng ghi
# đè lên nhau trong cùng cache -> file lỗi hoặc treo. Tải xong rồi mới fork ra
# worker, mỗi worker chỉ đọc từ đĩa.
with tracker.stage("download_weights", detail=MODEL_ID) as rec:
    MODEL_DIR = snapshot_download(MODEL_ID)
    rec.detail = MODEL_DIR
print("MODEL_DIR =", MODEL_DIR)

In [ ]:
WORKER_PATH = "/kaggle/working/aic_caption_worker.py"


def launch_workers(*, tag: str, limit: int = 0) -> list[dict[str, Any]]:
    """Mỗi GPU một process. `tag` tách hẳn file của lần chạy thử và lần chạy thật."""
    processes = []
    for shard in range(NUM_SHARDS):
        out_path = SHARD_DIR / f"{tag}_shard{shard}.jsonl"
        progress_path = SHARD_DIR / f"{tag}_progress{shard}.json"
        log_path = SHARD_DIR / f"{tag}_worker{shard}.log"
        command = [
            sys.executable, WORKER_PATH,
            "--tasks", str(TASKS_PATH),
            "--out", str(out_path),
            "--progress", str(progress_path),
            "--shard", str(shard),
            "--num-shards", str(NUM_SHARDS),
            "--model", str(MODEL_DIR),
            "--dtype", DTYPE,
            "--batch-size", str(BATCH_SIZE),
            "--max-visual-tokens", str(MAX_VISUAL_TOKENS),
            "--min-visual-tokens", str(MIN_VISUAL_TOKENS),
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--limit", str(limit),
        ]
        environment = dict(
            os.environ,
            CUDA_VISIBLE_DEVICES=str(shard),   # ghim process này vào đúng 1 GPU
            TOKENIZERS_PARALLELISM="false",    # tắt cảnh báo fork của tokenizers
            HF_HUB_OFFLINE="1",                # trọng số đã tải xong ở cell trên
        )
        handle = log_path.open("w", encoding="utf-8")
        processes.append({
            "shard": shard,
            "proc": subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT, env=environment),
            "log_handle": handle,
            "log_path": log_path,
            "progress": progress_path,
            "out": out_path,
        })
        append_progress_log(f"khởi động worker shard={shard} trên GPU {shard} -> {log_path.name}")
    return processes


def monitor(processes: list[dict[str, Any]], total: int, label: str) -> dict[str, int]:
    bar = tqdm(total=total, desc=label, unit="frame", dynamic_ncols=True, disable=not SHOW_TQDM)
    last_heartbeat = 0.0
    totals = {"done": 0, "found": 0, "failed": 0}
    try:
        while True:
            states = [read_json_safe(entry["progress"]) for entry in processes]
            totals = {
                "done": sum(state.get("done", 0) for state in states),
                "found": sum(state.get("found", 0) for state in states),
                "failed": sum(state.get("failed", 0) for state in states),
            }
            rates = [state["rate_sec_per_frame"] for state in states if state.get("rate_sec_per_frame")]
            remaining = max(0, total - totals["done"])
            eta_sec = remaining * (sum(rates) / len(rates)) / len(processes) if rates else 0.0

            bar.n = min(totals["done"], total)
            bar.set_postfix_str(
                f"ok={totals['found']} lỗi={totals['failed']} ETA={eta_sec / 60:.1f}m"
            )
            bar.refresh()

            alive = [entry for entry in processes if entry["proc"].poll() is None]
            now = time.monotonic()
            if now - last_heartbeat >= HEARTBEAT_INTERVAL_SEC:
                last_heartbeat = now
                append_progress_log(
                    f"{label}: {totals['done']}/{total} frame | ok={totals['found']} "
                    f"lỗi={totals['failed']} | worker còn sống={len(alive)} | ETA={eta_sec / 60:.1f} phút"
                )
                write_progress_state({
                    "status": "running", "current_task": label, "frame_total": total,
                    "frame_done": totals["done"], "found": totals["found"], "failed": totals["failed"],
                    "elapsed_sec": round(tracker.elapsed_total_sec(), 3), "eta_sec": round(eta_sec, 3),
                })
            if not alive:
                break
            time.sleep(POLL_INTERVAL_SEC)
    finally:
        bar.close()
        for entry in processes:
            entry["log_handle"].close()

    broken = [entry for entry in processes if entry["proc"].returncode != 0]
    for entry in broken:
        lines = entry["log_path"].read_text(encoding="utf-8", errors="replace").splitlines()
        print(f"\n===== worker shard={entry['shard']} thoát với mã {entry['proc'].returncode} =====")
        print("\n".join(lines[-30:]))
    if broken:
        raise RuntimeError(
            f"{len(broken)}/{len(processes)} worker hỏng (xem log ở trên). "
            "Kết quả đã chạy xong vẫn nằm trong shards/*.jsonl — chạy lại cell này sẽ tiếp tục, không làm lại từ đầu."
        )
    return totals

## 6. Chạy thử vài frame

Đọc caption sinh ra trước khi đốt hàng giờ GPU. Nếu output rỗng hoặc là ký tự
rác ở mọi frame, đó thường là fp16 tràn số trên Turing — đổi `DTYPE = "float32"`
ở mục 1 (3B fp32 ~12GB, vẫn vừa một T4 16GB, chậm hơn) rồi chạy lại.

In [ ]:
for stale in SHARD_DIR.glob("smoke_*"):
    stale.unlink()   # chạy lại mục này phải đọc lại từ đầu, không tính là "đã xong"

with tracker.stage("smoke_test", detail=f"{SMOKE_FRAMES} frame/GPU") as rec:
    shard_sizes = [len(tasks[shard::NUM_SHARDS]) for shard in range(NUM_SHARDS)]
    smoke_total = sum(min(SMOKE_FRAMES, size) for size in shard_sizes)
    smoke_totals = monitor(launch_workers(tag="smoke", limit=SMOKE_FRAMES), smoke_total, "Caption (thử)")
    rec.detail = f"{smoke_totals['done']} frame, có caption {smoke_totals['found']}"

smoke_rows = [row for shard in range(NUM_SHARDS)
              for row in read_jsonl(SHARD_DIR / f"smoke_shard{shard}.jsonl")]
for row in smoke_rows:
    print(f"\n--- {row['image_relpath']} (frame_idx={row['frame_idx']}, {row['status']}) ---")
    print("    caption:", row["caption"] or "(rỗng)")
    print("    tags   :", ", ".join(row["tags"]) or "(không có)")
    if row["error"]:
        print("    LỖI:", row["error"])

## 7. Chạy toàn bộ batch

Chạy lại cell này sau khi phiên chết sẽ **tiếp tục từ chỗ dở** — worker đọc lại
`shards/full_shard*.jsonl` và bỏ qua key đã xong.

In [ ]:
with tracker.stage("caption_all", detail=f"{len(tasks)} frame / {NUM_SHARDS} GPU") as rec:
    run_totals = monitor(launch_workers(tag="full"), len(tasks), "Caption")
    rec.detail = f"{run_totals['done']} frame, có caption {run_totals['found']}, lỗi {run_totals['failed']}"

print(run_totals)

## 8. Gộp shard thành manifest đúng contract

Mỗi frame cho ra tối đa hai `captions`: một `detailed` (câu mô tả) và một `tags`
(nhãn vật thể nối bằng dấu phẩy). Frame không sinh được caption nằm ở
`caption_frame_status.jsonl` với `status="empty"` — khác OCR, ảnh nào cũng phải
mô tả được, nên caption rỗng là **bất thường** chứ không phải kết quả hợp lệ.

In [ ]:
with tracker.stage("merge_shards") as rec:
    records: dict[str, dict[str, Any]] = {}
    for shard in range(NUM_SHARDS):
        shard_path = SHARD_DIR / f"full_shard{shard}.jsonl"
        if shard_path.exists():
            for row in read_jsonl(shard_path):
                records[row["key"]] = row   # chạy lại có thể ghi trùng key: giữ bản mới nhất
    records_list = sorted(records.values(), key=lambda row: (row["video_id"], row["frame_idx"]))

    caption_rows, status_rows = [], []
    per_video: dict[str, list[dict[str, Any]]] = defaultdict(list)
    excluded = Counter()
    for row in records_list:
        captions = []
        if row["caption"]:
            captions.append({"text": row["caption"], "language": "vi", "caption_type": "detailed"})
        if row["tags"]:
            captions.append({"text": ", ".join(row["tags"]), "language": "vi", "caption_type": "tags"})
        # Loại frame sai khoá join RA KHỎI manifest thay vì để assert vỡ ở cuối:
        # tới bước này GPU đã chạy xong rồi, đánh hỏng cả pack vì vài cái tên
        # file lạ là mất trắng hàng giờ. Manifest luôn hợp lệ, phần bị loại được
        # đếm và báo lại ở mục 9.
        reason = ""
        if not VIDEO_ID_RE.match(row["video_id"]):
            reason = "video_id không khớp ^L##_V###$"
        elif row["frame_idx"] < 0:
            reason = "không suy ra được frame_idx từ tên file"
        if reason:
            excluded[reason] += 1
        elif captions:
            manifest_row = {
                "video_id": row["video_id"],
                "frame_idx": row["frame_idx"],
                "captions": captions,
            }
            caption_rows.append(manifest_row)
            per_video[row["video_id"]].append(manifest_row)
        status_rows.append({
            "video_id": row["video_id"],
            "frame_idx": row["frame_idx"],
            "image_relpath": row["image_relpath"],
            "status": row["status"],
            "caption_chars": len(row["caption"]),
            "tag_count": len(row["tags"]),
            "manifest_excluded": reason,
            "error": row["error"],
        })

    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "caption_manifest.jsonl", caption_rows)
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "caption_frame_status.jsonl", status_rows)
    for video_id, rows in per_video.items():
        atomic_write_jsonl(OUTPUT_ROOT / "videos" / video_id / "caption_manifest.jsonl", rows)

    atomic_write_json(OUTPUT_ROOT / "model_info.json", {
        "component": "caption",
        "model": MODEL_ID,
        "pack_version": PACK_VERSION,
        "device": f"cuda x{NUM_SHARDS} ({DTYPE})",
        "prompt_version": "caption_tags_lines_v1",
        "batch": BATCH,
        "keyframe_root": str(KEYFRAME_ROOT),
        "max_visual_tokens": MAX_VISUAL_TOKENS,
        "objects_note": "nhãn vật thể nằm ở caption_type='tags'; không kèm bbox vì không đo được toạ độ thật",
    })
    atomic_write_json(OUTPUT_ROOT / "_SUCCESS.json", {
        "status": "success",
        "stage": STAGE_NAME,
        "video_count": len(per_video),
        "frame_count": len(status_rows),
        "runtime_sec": round(tracker.elapsed_total_sec(), 3),
        "created_at_utc": utc_now(),
    })
    rec.detail = f"{len(caption_rows)} frame có caption / {len(status_rows)} frame"

print(f"caption_manifest.jsonl     : {len(caption_rows)} dòng")
print(f"caption_frame_status.jsonl : {len(status_rows)} dòng")
for reason, count in excluded.items():
    print(f"[CẢNH BÁO] {count} frame bị loại khỏi manifest — {reason}")

In [ ]:
with tracker.stage("package_zip", detail=str(ZIP_PATH)):
    create_zip(OUTPUT_ROOT, ZIP_PATH)

write_progress_state({
    "status": "success", "frame_total": len(tasks), "frame_done": len(status_rows),
    "elapsed_sec": round(tracker.elapsed_total_sec(), 3), "eta_sec": 0.0,
})
print({"zip": str(ZIP_PATH), "MB": round(ZIP_PATH.stat().st_size / 1e6, 2)})

## 9. Kiểm tra output

In [ ]:
with tracker.stage("validate_output"):
    for row in caption_rows:
        assert VIDEO_ID_RE.match(row["video_id"]), f"video_id sai dạng: {row['video_id']}"
        assert row["frame_idx"] >= 0
        assert row["captions"]
        for caption in row["captions"]:
            assert caption["text"].strip()
            assert caption["caption_type"] in {"short", "detailed", "tags", "crop"}
    keys = {(row["video_id"], row["frame_idx"]) for row in caption_rows}
    assert len(keys) == len(caption_rows), "trùng (video_id, frame_idx) trong manifest"

empty = [row for row in status_rows if row["status"] == "empty"]
failed = [row for row in status_rows if row["status"] == "error"]
dropped = [row for row in status_rows if row["manifest_excluded"]]
missing = len(tasks) - len(status_rows)

lengths = sorted(row["caption_chars"] for row in status_rows if row["caption_chars"])
covered = defaultdict(lambda: [0, 0])
for row in status_rows:
    covered[row["video_id"]][0] += 1
    if row["caption_chars"]:
        covered[row["video_id"]][1] += 1

print(f"{'video_id':16s} {'frame':>7s} {'có caption':>12s} {'độ phủ':>8s}")
for video_id in sorted(covered):
    total_frames, with_caption = covered[video_id]
    print(f"{video_id:16s} {total_frames:7d} {with_caption:12d} {with_caption / total_frames:7.0%}")

if lengths:
    print(f"\nĐộ dài caption (ký tự): ngắn nhất {lengths[0]}, "
          f"trung vị {lengths[len(lengths) // 2]}, dài nhất {lengths[-1]}")
print(f"Tổng nhãn vật thể: {sum(row['tag_count'] for row in status_rows)}")
if empty:
    print(f"[CẢNH BÁO] {len(empty)} frame không sinh được caption, ví dụ: {[row['image_relpath'] for row in empty[:3]]}")
if failed:
    print(f"[CẢNH BÁO] {len(failed)} frame lỗi, ví dụ: {failed[:3]}")
if dropped:
    print(f"[CẢNH BÁO] {len(dropped)} frame bị loại khỏi manifest do sai khoá join "
          f"(caption vẫn nằm nguyên trong caption_frame_status.jsonl), ví dụ: "
          f"{[row['image_relpath'] for row in dropped[:3]]}")
if missing:
    print(f"[CẢNH BÁO] còn {missing} frame CHƯA xử lý — chạy lại mục 7 để tiếp tục.")
if not (empty or failed or dropped or missing):
    print("Validation passed — pack đầy đủ và đúng contract.")

## 10. Tổng kết thời gian

In [ ]:
print("=== Tác vụ pipeline ===")
print(f"{'Tác vụ':24s} {'Trạng thái':12s} {'Thời gian (s)':>14s}  chi tiết")
for record in tracker.records:
    print(f"{record.name:24s} {record.status:12s} {record.duration_sec:14.2f}  {record.detail}")
print(f"\nTổng thời gian toàn pipeline: {tracker.elapsed_total_sec() / 60:.2f} phút")

records = status_rows
if records:
    print(f"Tốc độ thực tế: {tracker.elapsed_total_sec() / len(records):.2f} s/frame "
          f"(đã tính cả thời gian tải model, trên {NUM_SHARDS} GPU song song)")
print(f"Chi tiết từng tác vụ: {STAGE_TIMINGS_PATH}")

if PROGRESS_LOG_PATH.exists():
    print("\n===== 15 dòng log cuối =====")
    print("\n".join(PROGRESS_LOG_PATH.read_text(encoding="utf-8").splitlines()[-15:]))